# SpeechSR V2 — Colab Training Notebook

Full pipeline for **ProposedModelV2** including:
- Optional CMRxRecon cardiac cine pretraining (replaces fastMRI)
- Generator pretraining with perceptual loss + patch-based data
- GAN fine-tuning with PatchGAN discriminator + k-space consistency
- Temporal inference for dynamic speech volumes
- Quick evaluation (PSNR / SSIM) with side-by-side visualisation

**Before running:** Enable a GPU runtime — Runtime → Change runtime type → T4 GPU (free tier).

See `IMPROVEMENTS.md` in the repo for full architecture and loss function details.

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/SpeechSR'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/marioknicola/SpeechSR.git {REPO_DIR}
    %cd {REPO_DIR}

# Force-reinstall scipy/skimage so they're built against Colab's numpy version.
# Without this, Colab's pre-installed binaries cause ABI mismatches.
!pip install -q --force-reinstall scipy scikit-image
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. Configuration

Edit the paths and flags below before running any training cells.

In [ ]:
# ── Data paths (on your Google Drive) ────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive'
INPUT_DIR    = f'{DRIVE_ROOT}/SpeechSR_data/Synth_LR'    # 128×128 LR NIfTI files
TARGET_DIR   = f'{DRIVE_ROOT}/SpeechSR_data/HR'          # 512×512 HR NIfTI files
DYNAMIC_DIR  = f'{DRIVE_ROOT}/SpeechSR_data/Dynamic_128' # real dynamic LR (no HR needed)
OUTPUT_DIR   = f'{DRIVE_ROOT}/SpeechSR_outputs'          # all checkpoints go here

# ── Subject split ─────────────────────────────────────────────────────────────
SUBJECTS     = '0021 0022 0023 0024 0025 0026 0027'
VAL_SUBJECT  = '0022'
TEST_SUBJECT = '0021'

# ── Training settings ─────────────────────────────────────────────────────────
PATCH_SIZE       = 64     # LR patch size (→ 512 HR patch at 4× scale)
BATCH_SIZE       = 2      # reduce to 1 if OOM on T4
NUM_WORKERS      = 2
PROPOSED_SIZE    = 512    # HR target size — 4× from 128 LR; matches real HR data

# ── Temporal mode ─────────────────────────────────────────────────────────────
# 3-channel input (t-1, t, t+1) is now the default for all proposed2 training.
# Static LR frames just repeat the single frame across all 3 channels, which
# is safe and primes the head for real temporal context from Dynamic_128.
USE_TEMPORAL     = True

# ── Dynamic auxiliary stream ──────────────────────────────────────────────────
# Real unlabelled dynamic LR data from Dynamic_128/ is used as a self-supervised
# stream: temporal consistency loss + k-space data consistency loss.
# No HR targets needed. Set USE_DYNAMIC=False to run supervised-only (ablation).
USE_DYNAMIC      = True
LAMBDA_TEMPORAL  = 0.10   # motion-weighted temporal consistency weight
LAMBDA_KSPACE_DYN = 0.05  # k-space data consistency weight on dynamic stream

# ── Stage 1: Generator pretraining ────────────────────────────────────────────
PRETRAIN_EPOCHS  = 150
PRETRAIN_LR      = 1e-4

# ── Stage 2: GAN fine-tuning (optional) ───────────────────────────────────────
# Set GAN_EPOCHS = 0 to stop after Stage 1 (recommended first run).
GAN_EPOCHS       = 0
GAN_LR           = 1e-4
LAMBDA_ADV       = 0.003  # lower than previous runs; previous 0.01 caused D collapse
LAMBDA_KSPACE    = 0.1
ALPHA_PERCEP     = 0.1

# ── Existing checkpoint to warm-start from (leave empty to train from scratch) ─
# After CMRxRecon pretraining (Section 2), this is set automatically.
# Set manually, e.g.: f'{OUTPUT_DIR}/cmrxrecon_pretrained/best_model.pth'
GENERATOR_CHECKPOINT = ''

import os
for d in [OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Configuration set.')
print(f'  Input:         {INPUT_DIR}')
print(f'  Target:        {TARGET_DIR}')
print(f'  Dynamic:       {DYNAMIC_DIR}')
print(f'  Output:        {OUTPUT_DIR}')
print(f'  PROPOSED_SIZE: {PROPOSED_SIZE}  (4× SR: 128 → 512)')
print(f'  Temporal:      {USE_TEMPORAL}')
print(f'  Dynamic aux:   {USE_DYNAMIC}')

---
## 2. (Optional) CMRxRecon Pretraining

Pretrain the generator on **CMRxRecon 2023 cardiac cine MRI** before fine-tuning on speech MRI.  
Pretraining is strongly recommended when you have only 5–7 training subjects.

**Why CMRxRecon (not fastMRI)?**
- Cardiac cine gives *genuine* temporal motion — (t-1, t, t+1) are different cardiac phases, not repeated statics.  
  The temporal consistency loss therefore trains on real dynamics from day one.
- Acquisition type (dynamic, undersampled, multi-coil) is much closer to speech MRI than brain single-coil MRI.
- Same k-space truncation / zero-padding pipeline applies directly.

**Access:** Register at https://www.synapse.org, accept the data use certificate for  
[syn51471091](https://www.synapse.org/#!Synapse:syn51471091), and generate a Personal Access Token (PAT).  
Run the *CMRxRecon — Download* cell below; it fetches only the files you need.

Skip this section entirely if you already have a pretrained checkpoint.

In [ ]:
# ── CMRxRecon paths and settings ─────────────────────────────────────────────
# The download cell (below) will populate CMRXRECON_DATA_DIR automatically.
# If you already downloaded the data, set the path manually here.
CMRXRECON_DATA_DIR   = f'{DRIVE_ROOT}/CMRxRecon/MultiCoil/Cine/TrainingSet/FullSample'
CMRXRECON_OUTPUT_DIR = f'{OUTPUT_DIR}/cmrxrecon_pretrained'

CMRXRECON_EPOCHS     = 30
CMRXRECON_BATCH_SIZE = 4
CMRXRECON_MAX_FILES  = 100  # 100 subjects × ~10 slices × ~14 frames ≈ 14k samples

# Synapse credentials — set your PAT here or export SYNAPSE_AUTH_TOKEN in the env
SYNAPSE_PAT = ''  # paste your token, e.g. 'eyJ0eXAiOiJK...'

import os
os.makedirs(CMRXRECON_OUTPUT_DIR, exist_ok=True)
print(f'CMRxRecon data dir : {CMRXRECON_DATA_DIR}')
print(f'CMRxRecon output   : {CMRXRECON_OUTPUT_DIR}')
print(f'Max files          : {CMRXRECON_MAX_FILES}')

In [ ]:
max_files_flag = f'--max-files {CMRXRECON_MAX_FILES}' if CMRXRECON_MAX_FILES else ''

!python pretrain_cmrxrecon.py \
    --data-dir        "{CMRXRECON_DATA_DIR}" \
    --output-dir      "{CMRXRECON_OUTPUT_DIR}" \
    --epochs          {CMRXRECON_EPOCHS} \
    --batch-size      {CMRXRECON_BATCH_SIZE} \
    --target-size     {PROPOSED_SIZE} \
    --in-channels     3 \
    --lambda-temporal {LAMBDA_TEMPORAL} \
    --lambda-kspace   {LAMBDA_KSPACE_DYN} \
    --alpha-percep    {ALPHA_PERCEP} \
    --num-workers     {NUM_WORKERS} \
    {max_files_flag}

# Warm-start speech MRI training from this checkpoint
GENERATOR_CHECKPOINT = f'{CMRXRECON_OUTPUT_DIR}/best_model.pth'
print(f'CMRxRecon pretrain done. Checkpoint: {GENERATOR_CHECKPOINT}')

In [ ]:
# ── CMRxRecon — Download ──────────────────────────────────────────────────────
# Downloads the CMRxRecon 2023 MultiCoil Cine FullSample training set from Synapse.
# Requires SYNAPSE_PAT to be set in the config cell above.
#
# Steps performed:
#   1. Install synapseclient (already bundled in newer Colab; force-install just in case)
#   2. Login with your Personal Access Token
#   3. Download up to CMRXRECON_MAX_FILES subject folders from:
#        syn51471091 → MultiCoil/Cine/TrainingSet/FullSample/
#      Each subject folder contains CineSAX.mat (short-axis) and CineLAX.mat (long-axis).
#   4. Files are saved to CMRXRECON_DATA_DIR on your mounted Drive.
#
# NOTE: Downloading 100 subjects is ~8–12 GB. This takes 15–30 min on a typical
#       Colab network connection. Reduce CMRXRECON_MAX_FILES to speed up.
# NOTE: You must have accepted the CMRxRecon data use agreement on Synapse before
#       your PAT will grant download access.

import os, subprocess, sys

if not SYNAPSE_PAT:
    raise ValueError(
        'Set SYNAPSE_PAT in the config cell (Section 2) before running this download cell.\n'
        'Generate a token at: https://www.synapse.org/#!PersonalAccessTokens:'
    )

# Install synapseclient if needed
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'synapseclient'])
import synapseclient
from synapseclient import Synapse

# Authenticate
syn = Synapse()
syn.login(authToken=SYNAPSE_PAT, silent=True)
print('Synapse login OK')

# Discover subject folders under FullSample (children of syn51471091 → subdirectories)
# CMRxRecon folder structure on Synapse:
#   syn51471091 (root)
#     └─ MultiCoil/
#         └─ Cine/
#             └─ TrainingSet/
#                 └─ FullSample/
#                     └─ P001/ → CineSAX.mat, CineLAX.mat
#                     └─ P002/ ...
#                     ...
CMRXRECON_SYNAPSE_ID = 'syn51471091'

print(f'Resolving children of {CMRXRECON_SYNAPSE_ID} ...')
all_children = list(syn.getChildren(CMRXRECON_SYNAPSE_ID, includeTypes=['folder']))
print(f'Found {len(all_children)} top-level entries.')

# Navigate to the FullSample folder (handles both flat and nested layouts)
def find_folder(syn_obj, name):
    for child in syn_obj.getChildren(syn_obj, includeTypes=['folder']):
        if child['name'].lower() == name.lower():
            return child['id']
    return None

# Walk down to FullSample if we landed at the root
target_id = CMRXRECON_SYNAPSE_ID
for subfolder in ['MultiCoil', 'Cine', 'TrainingSet', 'FullSample']:
    children = {c['name']: c['id'] for c in syn.getChildren(target_id, includeTypes=['folder'])}
    if subfolder in children:
        target_id = children[subfolder]
        print(f'  → {subfolder} ({target_id})')
    else:
        print(f'  Subfolder "{subfolder}" not found — trying current level directly')
        break

# Enumerate subject folders and download
subject_folders = list(syn.getChildren(target_id, includeTypes=['folder']))
subject_folders = subject_folders[:CMRXRECON_MAX_FILES]
print(f'\nDownloading {len(subject_folders)} subject folders to {CMRXRECON_DATA_DIR} ...')

os.makedirs(CMRXRECON_DATA_DIR, exist_ok=True)

for i, folder in enumerate(subject_folders):
    subj_id   = folder['id']
    subj_name = folder['name']
    subj_dir  = os.path.join(CMRXRECON_DATA_DIR, subj_name)
    os.makedirs(subj_dir, exist_ok=True)

    mat_files = list(syn.getChildren(subj_id, includeTypes=['file']))
    for f in mat_files:
        dest = os.path.join(subj_dir, f['name'])
        if os.path.exists(dest):
            continue  # resume-safe
        syn.get(f['id'], downloadLocation=subj_dir, ifcollision='overwrite.local')

    if (i + 1) % 10 == 0 or (i + 1) == len(subject_folders):
        print(f'  {i+1}/{len(subject_folders)} subjects done')

print(f'\nDownload complete. Data at: {CMRXRECON_DATA_DIR}')
GENERATOR_CHECKPOINT = ''   # will be set after pretraining below

---
## 3. GAN Training

Two-stage training on speech MRI subjects:
1. **Stage 1** — generator pretraining with `ForegroundEdgePerceptualLoss` (no discriminator)
2. **Stage 2** — adversarial fine-tuning with PatchGAN + k-space consistency

If you want to run them separately (e.g. inspect Stage 1 results first), set `GAN_EPOCHS = 0` for Stage 1 only, then re-run with `PRETRAIN_EPOCHS = 0` and the Stage 1 checkpoint.

In [ ]:
GAN_OUTPUT_DIR = f'{OUTPUT_DIR}/gan'

ckpt_flag    = f'--generator-checkpoint "{GENERATOR_CHECKPOINT}"' if GENERATOR_CHECKPOINT else ''
temporal_flag = '--temporal' if USE_TEMPORAL else ''
dynamic_flag  = f'--dynamic-dir "{DYNAMIC_DIR}" --lambda-temporal {LAMBDA_TEMPORAL} --lambda-kspace-dyn {LAMBDA_KSPACE_DYN}' if USE_DYNAMIC else '--no-dynamic'

!python train_gan.py \
    --input-dir           "{INPUT_DIR}" \
    --target-dir          "{TARGET_DIR}" \
    --output-dir          "{GAN_OUTPUT_DIR}" \
    --subjects            {SUBJECTS} \
    --val-subject         {VAL_SUBJECT} \
    --test-subject        {TEST_SUBJECT} \
    --proposed-target-size {PROPOSED_SIZE} \
    --patch-size          {PATCH_SIZE} \
    --augment \
    --pretrain-epochs     {PRETRAIN_EPOCHS} \
    --pretrain-lr         {PRETRAIN_LR} \
    --gan-epochs          {GAN_EPOCHS} \
    --gan-lr              {GAN_LR} \
    --lambda-adv          {LAMBDA_ADV} \
    --lambda-kspace       {LAMBDA_KSPACE} \
    --alpha-percep        {ALPHA_PERCEP} \
    --batch-size          {BATCH_SIZE} \
    --num-workers         {NUM_WORKERS} \
    {temporal_flag} \
    {dynamic_flag} \
    {ckpt_flag}

---
## 3b. (Alternative) Standard Training — no GAN

Trains ProposedModelV2 with `ForegroundEdgePerceptualLoss` only.  
Faster and more stable than GAN training, but will not recover fine texture as well.

In [ ]:
STANDARD_OUTPUT_DIR = f'{OUTPUT_DIR}/proposed2_standard'
STANDARD_EPOCHS     = 500

temporal_flag = '--temporal' if USE_TEMPORAL else ''
dynamic_flag  = f'--dynamic-dir "{DYNAMIC_DIR}" --lambda-temporal {LAMBDA_TEMPORAL} --lambda-kspace-dyn {LAMBDA_KSPACE_DYN}' if USE_DYNAMIC else '--no-dynamic'

!python train.py \
    --model               proposed2 \
    --input-dir           "{INPUT_DIR}" \
    --target-dir          "{TARGET_DIR}" \
    --output-dir          "{STANDARD_OUTPUT_DIR}" \
    --subjects            {SUBJECTS} \
    --val-subject         {VAL_SUBJECT} \
    --test-subject        {TEST_SUBJECT} \
    --proposed-target-size {PROPOSED_SIZE} \
    --patch-size          {PATCH_SIZE} \
    --augment \
    --perceptual-loss \
    --epochs              {STANDARD_EPOCHS} \
    --batch-size          {BATCH_SIZE} \
    --num-workers         {NUM_WORKERS} \
    {temporal_flag} \
    {dynamic_flag}

---
## 4. Inference

Run the best checkpoint on your test subject. Use `--temporal-window 3` for dynamic volumes to enforce frame-to-frame consistency.

In [ ]:
# Point to whichever checkpoint you want to use
GAN_OUTPUT_DIR   = f'{OUTPUT_DIR}/gan'
INFER_CHECKPOINT = f'{GAN_OUTPUT_DIR}/pretrained_generator.pth'
# INFER_CHECKPOINT = f'{GAN_OUTPUT_DIR}/best_gan_generator.pth'

INFER_INPUT      = f'{DRIVE_ROOT}/SpeechSR_data/Synth_LR/LR_kspace_Subject0021_oh.nii'
INFER_OUTPUT_DIR = f'{OUTPUT_DIR}/inference'
INFER_OUTPUT     = f'{INFER_OUTPUT_DIR}/proposed2_Subject0021_oh.nii'

# --temporal-window 3 is now the default in infer.py; set False only for 1-channel baselines
USE_TEMPORAL     = True

In [ ]:
import os
os.makedirs(INFER_OUTPUT_DIR, exist_ok=True)

temporal_flag = '--temporal-window 3' if USE_TEMPORAL else '--temporal-window 1'

!python infer.py \
    --model      proposed2 \
    --checkpoint "{INFER_CHECKPOINT}" \
    --input      "{INFER_INPUT}" \
    --output     "{INFER_OUTPUT}" \
    {temporal_flag}

print(f'Output saved to: {INFER_OUTPUT}')

---
## 5. Quick Evaluation

Computes PSNR and SSIM on the inference output vs the HR ground truth, then displays an input / SR / HR side-by-side comparison.

In [ ]:
EVAL_SR_PATH  = INFER_OUTPUT
EVAL_HR_PATH  = f'{DRIVE_ROOT}/SpeechSR_data/HR/HR_kspace_Subject0021_oh.nii'
EVAL_LR_PATH  = INFER_INPUT
EVAL_FRAME    = 0   # which frame (or slice) to display

In [ ]:
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F

def load_frame(path, frame=0):
    data = nib.load(path).get_fdata(dtype=np.float32)
    if data.ndim == 2:
        data = data[:, :, np.newaxis]
    img = data[:, :, min(frame, data.shape[2] - 1)]
    img = img - img.min()
    if img.max() > 0:
        img /= img.max()
    return img

def psnr_np(ref, est, data_range=1.0):
    mse = np.mean((ref - est) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(data_range ** 2 / mse)

def ssim_np(ref, est, data_range=1.0, win=11):
    C1, C2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    def pool(x): return np.array(F.avg_pool2d(
        torch.from_numpy(x).unsqueeze(0).unsqueeze(0), win, stride=1, padding=win//2).squeeze())
    mu1, mu2 = pool(ref), pool(est)
    s1  = pool(ref * ref) - mu1 * mu1
    s2  = pool(est * est) - mu2 * mu2
    s12 = pool(ref * est) - mu1 * mu2
    ssim_map = ((2*mu1*mu2 + C1) * (2*s12 + C2)) / ((mu1**2 + mu2**2 + C1) * (s1 + s2 + C2))
    return float(ssim_map.mean())

sr = load_frame(EVAL_SR_PATH, EVAL_FRAME)
hr = load_frame(EVAL_HR_PATH, EVAL_FRAME)

if sr.shape != hr.shape:
    sr_t = torch.from_numpy(sr).unsqueeze(0).unsqueeze(0)
    sr = F.interpolate(sr_t, size=hr.shape, mode='bilinear', align_corners=False).squeeze().numpy()

psnr_val = psnr_np(hr, sr)
ssim_val = ssim_np(hr, sr)
print(f'PSNR: {psnr_val:.2f} dB')
print(f'SSIM: {ssim_val:.4f}')

In [ ]:
import matplotlib.pyplot as plt

lr = load_frame(EVAL_LR_PATH, EVAL_FRAME)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes,
                           [lr, sr, hr],
                           ['LR Input', f'SR (proposed2)\nPSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}', 'HR Ground Truth']):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{INFER_OUTPUT_DIR}/comparison_Subject0021.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure saved.')

---
## 6. Batch Evaluation — All Phonemes

Runs inference and computes metrics across all 11 test phonemes and saves a metrics CSV.

In [ ]:
import os, json
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

PHONEMES      = ['ee', 'ff', 'mc', 'mm', 'mo', 'nn', 'oh', 'oo', 'sh', 'th', 'vv']
LR_TEMPLATE   = f'{DRIVE_ROOT}/SpeechSR_data/Synth_LR/LR_kspace_Subject0021_{{p}}.nii'
HR_TEMPLATE   = f'{DRIVE_ROOT}/SpeechSR_data/HR/HR_kspace_Subject0021_{{p}}.nii'
SR_OUTPUT_TPL = f'{INFER_OUTPUT_DIR}/proposed2_Subject0021_{{p}}.nii'

temporal_flag = '--temporal-window 3' if USE_TEMPORAL else '--temporal-window 1'
rows = []

for p in PHONEMES:
    lr_path = LR_TEMPLATE.format(p=p)
    hr_path = HR_TEMPLATE.format(p=p)
    sr_path = SR_OUTPUT_TPL.format(p=p)

    if not os.path.exists(lr_path):
        print(f'  Skipping {p} — LR file not found')
        continue

    os.system(f'python infer.py --model proposed2 --checkpoint "{INFER_CHECKPOINT}" '
              f'--input "{lr_path}" --output "{sr_path}" {temporal_flag}')

    if not os.path.exists(hr_path) or not os.path.exists(sr_path):
        continue

    sr = load_frame(sr_path)
    hr = load_frame(hr_path)
    if sr.shape != hr.shape:
        sr_t = torch.from_numpy(sr).unsqueeze(0).unsqueeze(0)
        sr = F.interpolate(sr_t, size=hr.shape, mode='bilinear', align_corners=False).squeeze().numpy()

    rows.append({'phoneme': p, 'psnr': psnr_np(hr, sr), 'ssim': ssim_np(hr, sr)})
    print(f'  {p}: PSNR={rows[-1]["psnr"]:.2f} dB  SSIM={rows[-1]["ssim"]:.4f}')

if rows:
    df = pd.DataFrame(rows)
    csv_path = f'{INFER_OUTPUT_DIR}/metrics_proposed2_Subject0021.csv'
    df.to_csv(csv_path, index=False)
    print(f'\nMean PSNR: {df["psnr"].mean():.2f} dB  |  Mean SSIM: {df["ssim"].mean():.4f}')
    print(f'Saved to: {csv_path}')

---
## 7. (Optional) Hyperparameter Optimisation

Uses Optuna to tune the loss weights and architecture hyperparameters for the standard (non-GAN) pipeline.  
Run this before GAN training to find good pixel-loss weights.

In [ ]:
HPO_OUTPUT_DIR = f'{OUTPUT_DIR}/hpo'
HPO_TRIALS     = 30
HPO_EPOCHS     = 25   # short runs per trial

!python train.py \
    --model               proposed2 \
    --input-dir           "{INPUT_DIR}" \
    --target-dir          "{TARGET_DIR}" \
    --output-dir          "{HPO_OUTPUT_DIR}" \
    --subjects            {SUBJECTS} \
    --val-subject         {VAL_SUBJECT} \
    --test-subject        {TEST_SUBJECT} \
    --patch-size          {PATCH_SIZE} \
    --augment \
    --perceptual-loss \
    --use-optuna \
    --n-trials            {HPO_TRIALS} \
    --hpo-epochs          {HPO_EPOCHS} \
    --train-after-hpo \
    --batch-size          {BATCH_SIZE} \
    --num-workers         {NUM_WORKERS}